# 03 — LSTM Point Forecast
**Project:** Probabilistic AQI Forecasting  

Baselines to beat:

| Model | RMSE | MAE |
|---|---|---|
| ARIMA(2,1,1) | 44.18 | 26.83 |
| Prophet | 59.31 | 45.90 |

Goals:
1. Build a sliding-window LSTM that forecasts next-step PM2.5
2. Train on hourly data (not daily — LSTM can handle it)
3. Evaluate on test set and compare against baselines
4. Save the trained model for reuse in notebook 04

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings, random, os
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

CITY    = 'Delhi'
TARGET  = 'PM2.5'
STEM    = f'../data/processed/{CITY.lower()}_aqi'

AQI_THRESHOLDS = {
    'Good': 30, 'Satisfactory': 60,
    'Moderate': 90, 'Poor': 120, 'Very Poor': 250
}

## 1. Load data

In [ ]:
train_df = pd.read_csv(f'{STEM}_train.csv', index_col='Datetime', parse_dates=True)
val_df   = pd.read_csv(f'{STEM}_val.csv',   index_col='Datetime', parse_dates=True)
test_df  = pd.read_csv(f'{STEM}_test.csv',  index_col='Datetime', parse_dates=True)

# Features: target + cyclical time encodings (no lag cols — LSTM handles memory itself)
FEATURES = [
    'PM2.5',
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
]

train_raw = train_df[FEATURES].dropna().values
val_raw   = val_df[FEATURES].dropna().values
test_raw  = test_df[FEATURES].dropna().values

print(f'Train: {train_raw.shape} | Val: {val_raw.shape} | Test: {test_raw.shape}')

## 2. Scale — fit on train only

In [ ]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw)   # fit here only
val_scaled   = scaler.transform(val_raw)
test_scaled  = scaler.transform(test_raw)

# PM2.5 is always the first column
TARGET_IDX = 0
print(f'Scaler fit on train. PM2.5 range: '
      f'{scaler.data_min_[TARGET_IDX]:.1f} – {scaler.data_max_[TARGET_IDX]:.1f} µg/m³')

## 3. Sliding-window Dataset

Each sample: **last `SEQ_LEN` hours → predict next hour's PM2.5**

In [ ]:
SEQ_LEN = 24   # look back 24 hours

class AQIDataset(Dataset):
    def __init__(self, data, seq_len, target_idx=0):
        self.X, self.y = [], []
        for i in range(len(data) - seq_len):
            self.X.append(data[i : i + seq_len])          # (seq_len, n_features)
            self.y.append(data[i + seq_len, target_idx])  # scalar
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


train_ds = AQIDataset(train_scaled, SEQ_LEN)
val_ds   = AQIDataset(val_scaled,   SEQ_LEN)
test_ds  = AQIDataset(test_scaled,  SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f'Train samples: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

## 4. LSTM model

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0.0,
            batch_first = True
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)        # out: (batch, seq, hidden)
        last    = out[:, -1, :]      # take last timestep
        return self.head(last).squeeze(-1)


model = LSTMForecaster(
    input_size  = len(FEATURES),
    hidden_size = 64,
    num_layers  = 2,
    dropout     = 0.2
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params:,}')

## 5. Training

In [ ]:
EPOCHS    = 30
LR        = 1e-3
PATIENCE  = 5      # early stopping

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5, verbose=True
)

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding grads
        optimizer.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)

    # ── Validate ──
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            pred = model(X_batch)
            val_batch_losses.append(criterion(pred, y_batch).item())
    val_loss = np.mean(val_batch_losses)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    # ── Early stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), '../models/lstm_best.pt')
    else:
        patience_counter += 1

    if epoch % 5 == 0 or patience_counter == PATIENCE:
        print(f'Epoch {epoch:3d} | train={train_loss:.5f} | val={val_loss:.5f} | patience={patience_counter}/{PATIENCE}')

    if patience_counter >= PATIENCE:
        print(f'Early stop at epoch {epoch}')
        break

print(f'\nBest val loss: {best_val_loss:.5f}')

## 6. Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train loss')
ax.plot(val_losses,   label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss (scaled)')
ax.set_title('LSTM training curve')
ax.legend()
plt.tight_layout()
plt.savefig('../visualizations/03_loss_curve.png', bbox_inches='tight')
plt.show()

## 7. Evaluate on test set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('../models/lstm_best.pt', map_location=DEVICE))
model.eval()

all_preds, all_actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        pred = model(X_batch.to(DEVICE)).cpu().numpy()
        all_preds.extend(pred)
        all_actuals.extend(y_batch.numpy())

# Inverse-scale PM2.5 only
def inverse_pm25(scaled_vals):
    """Undo MinMaxScaler on the PM2.5 column only."""
    mn = scaler.data_min_[TARGET_IDX]
    mx = scaler.data_max_[TARGET_IDX]
    return np.array(scaled_vals) * (mx - mn) + mn

preds_real   = inverse_pm25(all_preds).clip(min=0)
actuals_real = inverse_pm25(all_actuals)

rmse = np.sqrt(mean_squared_error(actuals_real, preds_real))
mae  = mean_absolute_error(actuals_real, preds_real)
mask = actuals_real > 5
mape = np.mean(np.abs((actuals_real[mask] - preds_real[mask]) / actuals_real[mask])) * 100

print('── LSTM Test Results ─────────────────────────────')
print(f'  RMSE : {rmse:.2f}  µg/m³   (baseline: 44.18)')
print(f'  MAE  : {mae:.2f}  µg/m³   (baseline: 26.83)')
print(f'  MAPE : {mape:.1f}%          (baseline: 27.9%)')
print()
if rmse < 44.18:
    print(f'✓ LSTM beats ARIMA by {44.18 - rmse:.2f} RMSE points')
else:
    print('✗ LSTM did not beat ARIMA — consider tuning SEQ_LEN or hidden_size')

## 8. Forecast plot

In [ ]:
# Build a time index for test predictions
test_index = test_df[FEATURES].dropna().index[SEQ_LEN:]

# Plot last 7 days (168 hours) for clarity
N = 168
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(test_index[-N:], actuals_real[-N:],
        color='#333', lw=1.3, label='Actual', zorder=3)
ax.plot(test_index[-N:], preds_real[-N:],
        color='#e07b54', lw=1.2, ls='--', label='LSTM forecast', zorder=2)

for name, thresh in AQI_THRESHOLDS.items():
    ax.axhline(thresh, lw=0.6, ls=':', color='tomato', alpha=0.5)
    ax.text(test_index[-1], thresh + 2, name, fontsize=7, color='tomato')

ax.set_title(f'LSTM point forecast — last 7 days of test set  (RMSE={rmse:.2f} µg/m³)')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../visualizations/03_lstm_forecast.png', bbox_inches='tight')
plt.show()

## 9. Save results + predictions

In [ ]:
# Save predictions for final comparison table
lstm_preds_series = pd.Series(preds_real, index=test_index, name='lstm')
lstm_preds_series.to_csv('../evaluation/lstm_test_preds.csv')

# Append to results table
try:
    results_df = pd.read_csv('../evaluation/baseline_results.csv', index_col='model')
except FileNotFoundError:
    results_df = pd.DataFrame(columns=['RMSE', 'MAE', 'MAPE'])
    results_df.index.name = 'model'

results_df.loc['LSTM'] = [round(rmse, 4), round(mae, 4), round(mape, 4)]
results_df.to_csv('../evaluation/baseline_results.csv')

print('Updated results table:')
print(results_df.round(2).to_string())
print()
print('Model weights saved to models/lstm_best.pt')
print('Next: notebook 04 — Quantile LSTM (the probabilistic layer)')